In [ ]:
pip install pandas numpy scipy statsmodels arch

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

# Фиксируем seed для воспроизводимости
np.random.seed(42)

# Загрузка факторов
factors = pd.read_csv('risk_factors.csv', index_col=0, parse_dates=True)


# Определяем, какие столбцы относятся к ставкам (RATE_PC), а какие к доходностям
rate_cols = [c for c in factors.columns if c.startswith('RATE_PC')]
return_cols = [c for c in factors.columns if c not in rate_cols]

print(f"Факторы ставок: {rate_cols}")
print(f"Факторы доходностей: {return_cols}")

# Функция для подгонки нормального и t-распределения, возвращает лучшую модель и параметры
def fit_distribution(series, dists=['norm', 't']):
    results = {}
    for d in dists:
        if d == 'norm':
            params = stats.norm.fit(series)
            loglik = stats.norm.logpdf(series, *params).sum()
            n_params = 2
        elif d == 't':
            params = stats.t.fit(series)
            loglik = stats.t.logpdf(series, *params).sum()
            n_params = 3
        aic = -2 * loglik + 2 * n_params
        results[d] = {'params': params, 'loglik': loglik, 'aic': aic}
    # Выбираем по минимальному AIC
    best = min(results, key=lambda x: results[x]['aic'])
    return best, results[best]['params'], results

# Функция для подгонки GARCH(1,1) с нормальным и t-распределением ошибок
def fit_garch(series, dists=['normal', 't']):
    results = {}
    for dist in dists:
        try:
            model = arch_model(series, vol='GARCH', p=1, q=1, dist=dist)
            res = model.fit(disp='off', show_warning=False)
            loglik = res.loglikelihood
            n_params = len(res.params)
            aic = -2 * loglik + 2 * n_params
            results[dist] = {'model': res, 'loglik': loglik, 'aic': aic, 'params': res.params}
        except Exception as e:
            print(f"Ошибка при подгонке GARCH для {series.name} с dist={dist}: {e}")
            results[dist] = None
    # Выбираем по минимальному AIC (игнорируем None)
    valid = {k:v for k,v in results.items() if v is not None}
    if not valid:
        return None, None, None
    best = min(valid, key=lambda x: valid[x]['aic'])
    return best, valid[best]['model'], valid

# Словари для хранения параметров
params_dict = {}

# 1. Обработка факторов ставок (простые распределения)
for col in rate_cols:
    series = factors[col].dropna()
    best, params, all_res = fit_distribution(series)
    params_dict[col] = {
        'model_type': 'distribution',
        'best_dist': best,
        'params': params,
        'aic': all_res[best]['aic'],
        'loglik': all_res[best]['loglik']
    }
    print(f"{col}: лучшая модель = {best}, AIC = {all_res[best]['aic']:.2f}, параметры = {params}")

# 2. Обработка факторов доходностей (GARCH)
for col in return_cols:
    series = factors[col].dropna()
    best, model, all_res = fit_garch(series)
    if best is not None:
        params_dict[col] = {
            'model_type': 'GARCH',
            'best_dist': best,
            'params': model.params.to_dict(),
            'aic': all_res[best]['aic'],
            'loglik': all_res[best]['loglik']
        }
        print(f"{col}: лучшая модель = GARCH-{best}, AIC = {all_res[best]['aic']:.2f}, параметры = {model.params.to_dict()}")
    else:
        # Если GARCH не сошелся, используем простое распределение (t или norm)
        print(f"GARCH не сошелся для {col}, используем простое распределение")
        best, params, all_res = fit_distribution(series)
        params_dict[col] = {
            'model_type': 'distribution',
            'best_dist': best,
            'params': params,
            'aic': all_res[best]['aic'],
            'loglik': all_res[best]['loglik']
        }
        print(f"{col}: лучшая модель = {best}, AIC = {all_res[best]['aic']:.2f}, параметры = {params}")

# Сохраняем параметры в DataFrame для удобства
rows = []
for factor, info in params_dict.items():
    row = {'factor': factor, 'model_type': info['model_type'], 'best_dist': info.get('best_dist', ''),
           'aic': info['aic'], 'loglik': info['loglik']}
    # Добавляем параметры в зависимости от типа
    if info['model_type'] == 'distribution':
        if info['best_dist'] == 'norm':
            row['mu'] = info['params'][0]
            row['sigma'] = info['params'][1]
        elif info['best_dist'] == 't':
            row['df'] = info['params'][0]
            row['loc'] = info['params'][1]
            row['scale'] = info['params'][2]
    elif info['model_type'] == 'GARCH':
        # Параметры GARCH: обычно ['mu', 'omega', 'alpha[1]', 'beta[1]'] + возможно 'nu' для t
        for k, v in info['params'].items():
            row[k] = v
    rows.append(row)

params_df = pd.DataFrame(rows).set_index('factor')
params_df.to_csv('model_params.csv')
print("\nПараметры моделей сохранены в model_params.csv")
print(params_df.round(4))